# Ppopy raw checkpoint continuation — 추론 전용

## 보존·진행 범위
- 로컬 실제 출력 **133개**, 직접 원문 대조 검수 **130개**를 `local_archive/`에 그대로 보존해요.
- **재학습하지 않아요.** 기존 `linawf/5epoch-qlora` Version 2의 epoch 1–4 어댑터를 SHA-256 대조 후 사용해요. Epoch 5는 없어요.
- 학습에 쓴 810건은 사용하지 않고 고정 `val_internal` **90건 × Base/E1/E2/E3/E4 = 450개**만 생성해요.
- 로컬에 없는 **317개를 우선 생성**하고 기존 133개도 각 GPU 작업 안에서 뒤이어 재생성해요.

## 비교 조건
로컬은 WSL vLLM/NF4/BF16, Kaggle T4는 Transformers+PEFT/NF4/FP16이에요. 서로 다른 실행 조건의 답변을 하나의 epoch 비교표로 섞지 않아요. Kaggle에서 450개를 동일 조건으로 다시 확보하고 로컬 원문·검수는 별도 보존해요. 동일 문장 여부는 해시로만 기록하며 로컬 PASS를 새 답변에 자동 승계하지 않아요.

교사 답변은 생성 입력에서 제외해요. greedy decoding, 768 출력 토큰 상한, 입력 무잘림, 고정 backbone revision과 토크나이저 입력 해시를 검증해요. GPU 2개면 Base와 어댑터 그룹을 나누어 추론해요. 각 답변 직후 JSONL을 저장하므로 실패 시 중간 산출물이 남아요.

## 결과 해석
**생성 완료는 의미 검수 완료가 아니에요.** 새 raw 답변을 실제로 읽어 입력·제품 표시·문헌과 연결한 뒤 논리 오류 발생 epoch와 semantic sweet spot을 판단해야 해요. 기존 loss 최저점은 epoch 2지만 답변 최적점으로 자동 확정하지 않아요. 사람 승인·독립 Cross-Model Critic·최종 test가 아니에요.


In [ ]:
# 기존 학습과 분리된 추론 전용 환경이에요. 학습은 실행하지 않아요.
import os,sys,subprocess,zipfile,hashlib,json
from pathlib import Path
os.environ.update(USE_TF="0",USE_FLAX="0",TOKENIZERS_PARALLELISM="false",HF_HUB_DISABLE_TELEMETRY="1")
subprocess.run([sys.executable,"-m","pip","install","transformers==4.57.3","peft==0.17.1","bitsandbytes==0.48.2","accelerate==1.11.0"],check=True)
manifests=list(Path("/kaggle/input").rglob("resume_manifest.json"))
if manifests:
    assert len(manifests)==1,manifests
    assert hashlib.sha256(manifests[0].read_bytes()).hexdigest()=="b7eb81a204422f209eaaa1fd8a8beb4f57cfb2a9201390da9e5d0dbe2d25612d"
    root=manifests[0].parent
    for name,sha in json.loads(manifests[0].read_text(encoding="utf-8"))["files"].items():
        p=(root/name).resolve();assert p.is_relative_to(root.resolve())
        assert hashlib.sha256(p.read_bytes()).hexdigest()==sha,name
    script=(root/"continue_raw.py").read_bytes()
else:
    zips=list(Path("/kaggle/input").rglob("ppopy_raw_resume_20261127.zip"))
    assert len(zips)==1,zips
    assert hashlib.sha256(zips[0].read_bytes()).hexdigest()=="eaffb1a61e2c1409f3c34c0453d53f1306ecb0d1dd5b949cf775605021cc52c6"
    with zipfile.ZipFile(zips[0]) as archive:
        script=archive.read("continue_raw.py")
path=Path("/kaggle/working/continue_raw.py");path.write_bytes(script)
print("재학습 없음. 로컬 133개 raw / 130개 검수 기록 보존. Kaggle 450개 생성.")
subprocess.run([sys.executable,str(path)],check=True)


In [ ]:
import json
from pathlib import Path
out=Path("/kaggle/working/ppopy_raw_continuation")
print((out/"generation_complete.json").read_text(encoding="utf-8"))
print("raw:",out/"raw_answers_kaggle.jsonl")
print("입력과 epoch별 원문:",out/"cases")
print("기존 로컬 기록:",out/"local_archive")
